In [10]:
import h5py as h5
import numpy as np
import pandas as pd
import os
from scipy.stats import linregress

In [ ]:
def process_ald(data_file_path):

    with h5.File(data_file_path, 'r') as hf:
        thickness_data = hf['measurement/ald_run/thickness_data'][:]
        growth_data = hf['measurement/ald_run/growth_rate'][:]
        settings_attr = hf['measurement/ald_run/settings'].attrs
        settings = {key: settings_attr[key] for key in settings_attr.keys()}
        settings['sample'] = hf['app/settings'].attrs['sample']

        temperature = hf['hardware/productivity_plc/settings'].attrs['PID_SetPoint']

    sample_name = settings['Sample_name']
    cycle_count = settings['Number_of_ALD_cycles']
    folder_path = os.path.dirname(data_file_path)
    filepath_txt = folder_path + '/' + sample_name + '.txt'

    # Import the data from a .txt file
    ellips_data = pd.read_csv(filepath_txt, delim_whitespace=True, skiprows=10) 
    thickness_ellips_data = ellips_data["Thick(nm).2"].to_numpy()
    fit_diff_data = ellips_data["Fit_Diff"].to_numpy()

    #Working with thickness data collected once per ALD cycle (from .h5)

    x = np.linspace(1, len(thickness_data), len(thickness_data))
    slope, _, r_sq, _, _ = linregress(x[10:], thickness_data[10:]) # skips first 10 cycles
    dep_rate_fit = slope
    growth_data_filtered = growth_data[growth_data > 0.1]
    dep_rate_mean = np.mean(growth_data_filtered)
    dep_rate_deviation = np.std(growth_data_filtered)
    thickness_start = thickness_ellips_data[0]
    thickness_end = thickness_ellips_data[-1]
    gpc_easy = (thickness_end - thickness_start)/cycle_count

    fit_diff_mean = np.mean(fit_diff_data)
    fit_diff_std = np.std(fit_diff_data)
    fit_diff_variation = fit_diff_std/fit_diff_mean*100

    print("Thickness line slope = {:.2f} nm/cycle with fitting R_sq = {:.4f}".format(dep_rate_fit, r_sq))
    print("Mean deposition rate = {:.2f} +- {:.2f} nm/cycle".format(dep_rate_mean, dep_rate_deviation))

    settings['total_MO_purge'] = settings['ALD_purge_time'] + settings['precursor_purge_time']
    settings['Initial_thickness'] = thickness_start
    settings['Temperature'] = temperature
    settings['dep_rate_fit'] = dep_rate_fit
    settings['dep_rate_mean'] = dep_rate_mean
    settings['dep_rate_direct'] = gpc_easy
    settings['dep_rate_deviation'] = dep_rate_deviation
    settings['fit_diff_variation'] = fit_diff_variation


    settings_to_write = pd.DataFrame(settings, index=[settings['run_id']])

    return settings_to_write

row = process_ald(r"C:\Users\lab\Documents\ALDBot Data\Wafer mv9c randomized exps\2025_02_07_time_11_16_40_Wafer_mv9c_randomized_dep1\250207_111640_ald_run.h5")

Thickness line slope = 0.15 nm/cycle with fitting R_sq = 1.0000
Mean deposition rate = 0.15 +- 0.01 nm/cycle


C:\Users\lab\AppData\Local\Temp\ipykernel_27680\1216608723.py:18: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ellips_data = pd.read_csv(filepath_txt, delim_whitespace=True, skiprows=10)


In [20]:
row.columns

Index(['ALD_prep_time', 'ALD_purge_flow_rate', 'ALD_purge_plasma_flow_rate',
       'ALD_purge_time', 'Ar_plasma_flow_rate', 'Ar_process_flow_rate',
       'H2_plasma_flow_rate', 'LC_preset', 'N2_plasma_flow_rate',
       'Number_of_ALD_cycles', 'RF_power_setpoint', 'Sample_name', 'TC_preset',
       'activation', 'ald_valves_delay', 'plasma_duration', 'plasma_prep_time',
       'plasma_pressure', 'plasma_purge_time', 'precursor_dose_time',
       'precursor_purge_time', 'process_pressure', 'profile', 'progress',
       'run_description', 'run_id', 'run_state', 'sample', 'total_MO_purge',
       'Initial_thickness', 'Temperature', 'dep_rate_fit', 'dep_rate_mean',
       'dep_rate_direct', 'dep_rate_deviation', 'fit_diff_variation'],
      dtype='object')

In [22]:
row['sample']

0sx6bgca69tg9000wkc43qjtec    0sx5afvfbxv4q000nrhv1hmv9c
Name: sample, dtype: object